# Отчёт по лабораторной работе №6

## Разработка клиента под Android

**Дисциплина:** Разработка кроссплатформенных приложений, семестр 2

**Выполнил:** студент группы 12002408

Блинов Кирилл Игоревич

**Проверил:** ассистент кафедры информационных и робототехнических систем

Пузырев Никита Сергеевич

<br>
<br>

**Сайт курсового проекта:** [https://hki-stroitel.ru/](https://hki-stroitel.ru/)


<p align="justify"><strong>Цель работы:</strong> получить практические навыки работы с Spring Framework и разработать клиент под Android.</p>

<h2 align="center">Задание</h2>

1. Изучить теоретический материал.
2. Написать приложение, следуя примеру из методических указаний.
3. Сделать отчёт.


## Ход работы

### 1. Настройка среды и подключение к серверу 5-й лабораторной

Для разработки и проверки приложения настроены Android Studio, Android SDK (API 34, `platform-tools`, `build-tools;34.0.0`) и виртуальное устройство **Pixel 7 API 30** (`system-images;android-30;google_apis;x86_64`).

В качестве серверной части используется Spring Boot приложение из лабораторной работы №5 (`sem2/lab5`), запущенное через Gradle на порту `8085`. Из Android-эмулятора локальный сервер хоста доступен по адресу `http://10.0.2.2:8085/`.

В проекте `android-client` реализованы три экрана с максимально простым интерфейсом и минимальным кодом:
- `LoginActivity` – форма входа (`POST /api/auth/login` с предварительным получением CSRF-токена `GET /api/auth/csrf`) и кнопка перехода к расписанию;
- `TaskManagerActivity` – загрузка списка задач `GET /api/tasks` через Retrofit и вывод в `RecyclerView`;
- `ScheduleWebActivity` – встроенный `WebView`, который запрашивает расписание `GET /api/lessons` с сервера 5-й лабораторной и отображает карточки занятий.


### 2. Зависимости и сетевая конфигурация

Файл `android-client/app/build.gradle.kts`:

```kotlin
plugins {
    id("com.android.application")
}

android {
    namespace = "com.example.raspisaclient"
    compileSdk = 34

    defaultConfig {
        applicationId = "com.example.raspisaclient"
        minSdk = 24
        targetSdk = 34
        versionCode = 1
        versionName = "1.0"
    }

    buildTypes {
        release {
            isMinifyEnabled = false
        }
    }

    compileOptions {
        sourceCompatibility = JavaVersion.VERSION_17
        targetCompatibility = JavaVersion.VERSION_17
    }
}

dependencies {
    implementation("androidx.appcompat:appcompat:1.7.0")
    implementation("com.google.android.material:material:1.12.0")
    implementation("androidx.constraintlayout:constraintlayout:2.1.4")
    implementation("com.squareup.retrofit2:retrofit:2.9.0")
    implementation("com.squareup.retrofit2:converter-gson:2.9.0")
    implementation("androidx.recyclerview:recyclerview:1.3.2")
}
```

Файл `app/src/main/res/xml/network_security_config.xml` (разрешение HTTP для `10.0.2.2`):

```xml
<?xml version="1.0" encoding="utf-8"?>
<network-security-config>
    <domain-config cleartextTrafficPermitted="true">
        <domain includeSubdomains="true">10.0.2.2</domain>
        <domain includeSubdomains="true">localhost</domain>
    </domain-config>
</network-security-config>
```

Файл `app/src/main/AndroidManifest.xml`:

```xml
<?xml version="1.0" encoding="utf-8"?>
<manifest xmlns:android="http://schemas.android.com/apk/res/android">

    <uses-permission android:name="android.permission.INTERNET" />

    <application
        android:allowBackup="true"
        android:label="@string/app_name"
        android:networkSecurityConfig="@xml/network_security_config"
        android:supportsRtl="true"
        android:theme="@style/Theme.RaspisaClient">

        <activity
            android:name=".LoginActivity"
            android:exported="true">
            <intent-filter>
                <action android:name="android.intent.action.MAIN" />
                <category android:name="android.intent.category.LAUNCHER" />
            </intent-filter>
        </activity>

        <activity android:name=".TaskManagerActivity" />
        <activity android:name=".ScheduleWebActivity" />
    </application>
</manifest>
```


### 3. Макеты интерфейса

`activity_login.xml`:

```xml
<?xml version="1.0" encoding="utf-8"?>
<LinearLayout xmlns:android="http://schemas.android.com/apk/res/android"
    android:layout_width="match_parent"
    android:layout_height="match_parent"
    android:orientation="vertical"
    android:padding="16dp">

    <EditText
        android:id="@+id/username"
        android:layout_width="match_parent"
        android:layout_height="wrap_content"
        android:hint="Логин"
        android:text="admin" />

    <EditText
        android:id="@+id/password"
        android:layout_width="match_parent"
        android:layout_height="wrap_content"
        android:hint="Пароль"
        android:inputType="textPassword"
        android:text="Admin!2026" />

    <Button
        android:id="@+id/loginButton"
        android:layout_width="match_parent"
        android:layout_height="wrap_content"
        android:layout_marginTop="12dp"
        android:text="Войти" />

    <Button
        android:id="@+id/openSiteButton"
        android:layout_width="match_parent"
        android:layout_height="wrap_content"
        android:layout_marginTop="8dp"
        android:text="Открыть сайт расписания" />
</LinearLayout>
```

`activity_task_manager.xml`:

```xml
<?xml version="1.0" encoding="utf-8"?>
<LinearLayout xmlns:android="http://schemas.android.com/apk/res/android"
    android:layout_width="match_parent"
    android:layout_height="match_parent"
    android:orientation="vertical"
    android:padding="12dp">

    <Button
        android:id="@+id/siteButton"
        android:layout_width="match_parent"
        android:layout_height="wrap_content"
        android:text="Открыть сайт расписания" />

    <androidx.recyclerview.widget.RecyclerView
        android:id="@+id/recyclerView"
        android:layout_width="match_parent"
        android:layout_height="0dp"
        android:layout_marginTop="8dp"
        android:layout_weight="1" />
</LinearLayout>
```

`item_task.xml`:

```xml
<?xml version="1.0" encoding="utf-8"?>
<LinearLayout xmlns:android="http://schemas.android.com/apk/res/android"
    android:layout_width="match_parent"
    android:layout_height="wrap_content"
    android:orientation="vertical"
    android:padding="12dp">

    <TextView
        android:id="@+id/titleTextView"
        android:layout_width="wrap_content"
        android:layout_height="wrap_content"
        android:textSize="16sp"
        android:textStyle="bold" />

    <TextView
        android:id="@+id/descriptionTextView"
        android:layout_width="wrap_content"
        android:layout_height="wrap_content"
        android:layout_marginTop="4dp"
        android:textSize="14sp" />
</LinearLayout>
```


### 4. Исходный код Android-клиента

`ApiClient.java`:

```java
package com.example.raspisaclient;

import java.util.ArrayList;
import java.util.HashMap;
import java.util.List;
import java.util.Map;
import okhttp3.Cookie;
import okhttp3.CookieJar;
import okhttp3.HttpUrl;
import okhttp3.OkHttpClient;
import retrofit2.Retrofit;
import retrofit2.converter.gson.GsonConverterFactory;

public class ApiClient {
    public static final String BASE_URL = "http://10.0.2.2:8085/";
    private static TaskApi api;

    public static TaskApi getApi() {
        if (api == null) {
            Map<String, List<Cookie>> store = new HashMap<>();
            OkHttpClient client = new OkHttpClient.Builder()
                    .cookieJar(new CookieJar() {
                        @Override
                        public void saveFromResponse(HttpUrl url, List<Cookie> cookies) {
                            store.put(url.host(), cookies);
                        }

                        @Override
                        public List<Cookie> loadForRequest(HttpUrl url) {
                            List<Cookie> cookies = store.get(url.host());
                            return cookies != null ? cookies : new ArrayList<>();
                        }
                    })
                    .build();

            Retrofit retrofit = new Retrofit.Builder()
                    .baseUrl(BASE_URL)
                    .client(client)
                    .addConverterFactory(GsonConverterFactory.create())
                    .build();

            api = retrofit.create(TaskApi.class);
        }
        return api;
    }
}
```

`TaskApi.java`:

```java
package com.example.raspisaclient;

import java.util.List;
import java.util.Map;
import retrofit2.Call;
import retrofit2.http.Body;
import retrofit2.http.GET;
import retrofit2.http.Header;
import retrofit2.http.POST;

public interface TaskApi {
    @GET("/api/auth/csrf")
    Call<Map<String, String>> getCsrf();

    @POST("/api/auth/login")
    Call<Map<String, Object>> login(
            @Header("X-CSRF-TOKEN") String csrfToken,
            @Body Map<String, String> body
    );

    @GET("/api/tasks")
    Call<List<Task>> getTasks();
}
```

`Task.java`:

```java
package com.example.raspisaclient;

public class Task {
    private Long id;
    private String title;
    private String description;
    private String status;
    private String category;

    public Long getId() {
        return id;
    }

    public String getTitle() {
        return title;
    }

    public String getDescription() {
        return description;
    }

    public String getStatus() {
        return status;
    }

    public String getCategory() {
        return category;
    }
}
```

`LoginActivity.java`:

```java
package com.example.raspisaclient;

import android.content.Intent;
import android.os.Bundle;
import android.widget.Button;
import android.widget.EditText;
import android.widget.Toast;
import androidx.appcompat.app.AppCompatActivity;
import java.util.HashMap;
import java.util.Map;
import retrofit2.Call;
import retrofit2.Callback;
import retrofit2.Response;

public class LoginActivity extends AppCompatActivity {
    private EditText usernameField;
    private EditText passwordField;

    @Override
    protected void onCreate(Bundle savedInstanceState) {
        super.onCreate(savedInstanceState);
        setContentView(R.layout.activity_login);

        usernameField = findViewById(R.id.username);
        passwordField = findViewById(R.id.password);
        Button loginButton = findViewById(R.id.loginButton);
        Button openSiteButton = findViewById(R.id.openSiteButton);

        loginButton.setOnClickListener(v -> doLogin());
        openSiteButton.setOnClickListener(v ->
                startActivity(new Intent(LoginActivity.this, ScheduleWebActivity.class)));
    }

    private void doLogin() {
        String username = usernameField.getText().toString().trim();
        String password = passwordField.getText().toString();
        TaskApi api = ApiClient.getApi();

        api.getCsrf().enqueue(new Callback<Map<String, String>>() {
            @Override
            public void onResponse(Call<Map<String, String>> call, Response<Map<String, String>> res) {
                if (!res.isSuccessful() || res.body() == null) {
                    Toast.makeText(LoginActivity.this, "Ошибка CSRF", Toast.LENGTH_SHORT).show();
                    return;
                }
                String token = res.body().get("token");
                Map<String, String> body = new HashMap<>();
                body.put("login", username);
                body.put("password", password);

                api.login(token, body).enqueue(new Callback<Map<String, Object>>() {
                    @Override
                    public void onResponse(Call<Map<String, Object>> call, Response<Map<String, Object>> loginRes) {
                        if (loginRes.isSuccessful()) {
                            startActivity(new Intent(LoginActivity.this, TaskManagerActivity.class));
                        } else {
                            Toast.makeText(LoginActivity.this, "Неверный логин или пароль", Toast.LENGTH_SHORT).show();
                        }
                    }

                    @Override
                    public void onFailure(Call<Map<String, Object>> call, Throwable t) {
                        Toast.makeText(LoginActivity.this, "Ошибка сети: " + t.getMessage(), Toast.LENGTH_SHORT).show();
                    }
                });
            }

            @Override
            public void onFailure(Call<Map<String, String>> call, Throwable t) {
                Toast.makeText(LoginActivity.this, "Сервер недоступен: " + t.getMessage(), Toast.LENGTH_SHORT).show();
            }
        });
    }
}
```

`TaskManagerActivity.java`:

```java
package com.example.raspisaclient;

import android.content.Intent;
import android.os.Bundle;
import android.widget.Button;
import android.widget.Toast;
import androidx.appcompat.app.AppCompatActivity;
import androidx.recyclerview.widget.LinearLayoutManager;
import androidx.recyclerview.widget.RecyclerView;
import java.util.List;
import retrofit2.Call;
import retrofit2.Callback;
import retrofit2.Response;

public class TaskManagerActivity extends AppCompatActivity {
    private RecyclerView recyclerView;

    @Override
    protected void onCreate(Bundle savedInstanceState) {
        super.onCreate(savedInstanceState);
        setContentView(R.layout.activity_task_manager);

        recyclerView = findViewById(R.id.recyclerView);
        recyclerView.setLayoutManager(new LinearLayoutManager(this));

        Button siteButton = findViewById(R.id.siteButton);
        siteButton.setOnClickListener(v ->
                startActivity(new Intent(TaskManagerActivity.this, ScheduleWebActivity.class)));

        loadTasks();
    }

    private void loadTasks() {
        ApiClient.getApi().getTasks().enqueue(new Callback<List<Task>>() {
            @Override
            public void onResponse(Call<List<Task>> call, Response<List<Task>> response) {
                if (response.isSuccessful() && response.body() != null) {
                    recyclerView.setAdapter(new TaskAdapter(response.body()));
                } else {
                    Toast.makeText(TaskManagerActivity.this, "Ошибка загрузки задач", Toast.LENGTH_SHORT).show();
                }
            }

            @Override
            public void onFailure(Call<List<Task>> call, Throwable t) {
                Toast.makeText(TaskManagerActivity.this, "Ошибка сети: " + t.getMessage(), Toast.LENGTH_SHORT).show();
            }
        });
    }
}
```

`TaskAdapter.java`:

```java
package com.example.raspisaclient;

import android.view.LayoutInflater;
import android.view.View;
import android.view.ViewGroup;
import android.widget.TextView;
import androidx.annotation.NonNull;
import androidx.recyclerview.widget.RecyclerView;
import java.util.List;

public class TaskAdapter extends RecyclerView.Adapter<TaskAdapter.TaskViewHolder> {
    private final List<Task> taskList;

    public TaskAdapter(List<Task> taskList) {
        this.taskList = taskList;
    }

    @NonNull
    @Override
    public TaskViewHolder onCreateViewHolder(@NonNull ViewGroup parent, int viewType) {
        View view = LayoutInflater.from(parent.getContext())
                .inflate(R.layout.item_task, parent, false);
        return new TaskViewHolder(view);
    }

    @Override
    public void onBindViewHolder(@NonNull TaskViewHolder holder, int position) {
        Task task = taskList.get(position);
        String title = task.getTitle() != null ? task.getTitle() : "";
        String status = task.getStatus() != null ? " [" + task.getStatus() + "]" : "";
        holder.titleTextView.setText(title + status);
        holder.descriptionTextView.setText(task.getDescription() != null ? task.getDescription() : "");
    }

    @Override
    public int getItemCount() {
        return taskList.size();
    }

    public static class TaskViewHolder extends RecyclerView.ViewHolder {
        TextView titleTextView;
        TextView descriptionTextView;

        public TaskViewHolder(@NonNull View itemView) {
            super(itemView);
            titleTextView = itemView.findViewById(R.id.titleTextView);
            descriptionTextView = itemView.findViewById(R.id.descriptionTextView);
        }
    }
}
```

`ScheduleWebActivity.java`:

```java
package com.example.raspisaclient;

import android.annotation.SuppressLint;
import android.os.Bundle;
import android.webkit.WebSettings;
import android.webkit.WebView;
import android.webkit.WebViewClient;
import androidx.appcompat.app.AppCompatActivity;

public class ScheduleWebActivity extends AppCompatActivity {
    private static final String HTML =
            "<!doctype html><html><head><meta charset='utf-8'>"
            + "<meta name='viewport' content='width=device-width,initial-scale=1'>"
            + "<style>body{font-family:sans-serif;margin:12px;background:#f7f7f7}"
            + "h2{margin:0 0 12px} .card{background:#fff;padding:10px;margin-bottom:8px;border-radius:6px;border:1px solid #ddd}"
            + ".t{font-weight:bold;font-size:15px} .m{color:#555;font-size:13px;margin-top:4px}</style></head><body>"
            + "<h2>Расписание занятий</h2><div id='list'>Загрузка...</div>"
            + "<script>"
            + "var xhr = new XMLHttpRequest();"
            + "xhr.open('GET', '/api/lessons', true);"
            + "xhr.onload = function() {"
            + "  var data = JSON.parse(xhr.responseText);"
            + "  var html = '';"
            + "  for (var i = 0; i < Math.min(data.length, 15); i++) {"
            + "    var l = data[i];"
            + "    html += '<div class=\"card\"><div class=\"t\">' + (l.studioName || l.title || 'Занятие') + '</div>'"
            + "      + '<div class=\"m\">' + l.dayOfWeek + ' · ' + (l.time || '') + ' · Группа ' + (l.groupNumber || '-') + '</div>'"
            + "      + '<div class=\"m\">' + (l.teacherName || '') + '</div></div>';"
            + "  }"
            + "  document.getElementById('list').innerHTML = html || 'Нет занятий';"
            + "};"
            + "xhr.onerror = function() { document.getElementById('list').innerHTML = 'Ошибка загрузки'; };"
            + "xhr.send();"
            + "</script></body></html>";

    @SuppressLint("SetJavaScriptEnabled")
    @Override
    protected void onCreate(Bundle savedInstanceState) {
        super.onCreate(savedInstanceState);
        setContentView(R.layout.activity_schedule_web);

        WebView webView = findViewById(R.id.webView);
        WebSettings settings = webView.getSettings();
        settings.setJavaScriptEnabled(true);
        settings.setDomStorageEnabled(true);
        webView.setWebViewClient(new WebViewClient());
        webView.loadDataWithBaseURL(ApiClient.BASE_URL, HTML, "text/html", "UTF-8", null);
    }
}
```


### 5. Проверка работы на эмуляторе

Приложение собрано командой `.\gradlew.bat assembleDebug` и установлено на эмулятор `Pixel_7_API_30`.

Экран авторизации (`LoginActivity`):

![Экран входа](images/01_login.png)

Список задач, полученный от Spring REST API (`/api/tasks`) после входа:

![Список задач](images/02_tasks.png)

Экран расписания занятий (`WebView` с запросом к `/api/lessons` сервера 5-й лабораторной):

![Расписание занятий](images/03_schedule_site.png)

**Вывод:** получил практические навыки работы с Spring Framework и разработал клиент под Android.
